# Notebook 5: Tensor Networks, DMRG & Time Evolution

This notebook covers Qforge's tensor network simulation backends for reaching **50-1000+ qubits** — far beyond exact state vector limits.

1. **MPS** (Matrix Product States) — simulate low-entanglement circuits with 50+ qubits
2. **DMRG** — find ground states of 1D Hamiltonians (Heisenberg, Ising, XXZ, custom)
3. **Custom Hamiltonians** — build arbitrary MPO Hamiltonians with `MPOBuilder`
4. **Excited States** — find multiple eigenstates via the penalty method
5. **TEBD** — real-time and imaginary-time evolution on finite MPS
6. **iTEBD** — ground states and dynamics of infinite 1D systems
7. **Distributed Simulation** — MPI-sharded state vectors for 30-40 qubits

In [3]:
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Matrix Product States (MPS)

An MPS represents an n-qubit state as a chain of small tensors instead of a single 2^n vector. For states with limited entanglement (e.g., ground states of local Hamiltonians), this is exponentially more efficient.

**Key parameter**: bond dimension `chi` — controls the maximum entanglement the MPS can represent. Larger chi = more accurate but slower.

| chi | Entanglement capacity | Typical use |
|-----|----------------------|-------------|
| 1   | Product states only  | Classical states |
| 2-8 | Low entanglement     | Shallow circuits, GHZ |
| 16-64 | Moderate           | DMRG ground states |
| 128-512 | High             | Critical systems, deep circuits |

In [5]:
from qforge.mps import MatrixProductState
from qforge.gates import H, CNOT, RY, RZ, X

# Create a 50-qubit MPS (impossible with state vectors!)
psi = MatrixProductState(n_qubits=50, max_bond_dim=64)

# Build a GHZ state: H on qubit 0, then CNOT cascade
H(psi, 0)
for i in range(49):
    CNOT(psi, i, i + 1)

print(f"50-qubit GHZ state created")
print(f"Bond dimensions: {psi.bond_dimensions()[:5]}... (all should be 2)")
print(f"Entanglement entropy at bond 0: {psi.entanglement_entropy(0):.4f} bits")
print(f"Max entanglement: {psi.max_entanglement():.4f} bits")
print(f"Norm: {psi.norm():.8f}")

50-qubit GHZ state created
Bond dimensions: [2, 2, 2, 2, 2]... (all should be 2)


ValueError: vector

In [ ]:
# Entanglement entropy profile across all bonds
n = 50
entropies = [psi.entanglement_entropy(b) for b in range(n - 1)]

plt.figure(figsize=(10, 4))
plt.plot(range(n - 1), entropies, 'o-', markersize=3)
plt.xlabel('Bond index')
plt.ylabel('Entanglement entropy (bits)')
plt.title('GHZ State: Entanglement Across 50 Qubits')
plt.ylim(-0.1, 1.5)
plt.grid(True, alpha=0.3)
plt.show()

---
## 2. DMRG — Ground State Finding

DMRG (Density Matrix Renormalization Group) is the gold standard for finding ground states of 1D quantum systems. It variationally optimizes an MPS to minimize the energy.

Qforge provides built-in Hamiltonians and supports both **two-site** and **single-site** DMRG algorithms.

In [ ]:
from qforge.dmrg import DMRG

# Transverse-field Ising model: H = -J * sum ZZ - h * sum X
# At h/J = 1 (critical point), the system has a quantum phase transition
n_sites = 20
dmrg = DMRG.ising(n_sites=n_sites, J=1.0, h=1.0, max_bond_dim=32)

energy, psi_gs = dmrg.run(n_sweeps=20, verbose=True, algorithm='2site')
print(f"\nGround state energy: {energy:.8f}")
print(f"Energy per site:     {energy / n_sites:.8f}")
print(f"Max bond dimension:  {max(psi_gs.bond_dimensions())}")

In [ ]:
# Magnetization profile: <Z_i> at each site
mag = dmrg.magnetization_profile()

plt.figure(figsize=(10, 4))
plt.bar(range(n_sites), mag, color='steelblue', alpha=0.8)
plt.xlabel('Site index')
plt.ylabel(r'$\langle Z_i \rangle$')
plt.title(f'Ising Ground State Magnetization (h/J=1.0, N={n_sites})')
plt.grid(True, alpha=0.3)
plt.show()

### 2a. Quantum Phase Transition

Sweep the transverse field `h` from 0 to 2 and observe the phase transition at `h/J = 1`.

In [ ]:
# Phase transition: sweep h/J and measure ground state properties
n_sites = 12
h_values = np.linspace(0.1, 2.5, 15)
energies_per_site = []
mid_entropy = []

for h_val in h_values:
    d = DMRG.ising(n_sites=n_sites, J=1.0, h=h_val, max_bond_dim=32)
    e, gs = d.run(n_sweeps=15)
    energies_per_site.append(e / n_sites)
    mid_entropy.append(gs.entanglement_entropy(n_sites // 2 - 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(h_values, energies_per_site, 'o-', color='navy')
ax1.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='Critical point')
ax1.set_xlabel('h / J')
ax1.set_ylabel('Energy per site')
ax1.set_title('Ground State Energy vs Transverse Field')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(h_values, mid_entropy, 's-', color='darkgreen')
ax2.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='Critical point')
ax2.set_xlabel('h / J')
ax2.set_ylabel('Entanglement entropy (bits)')
ax2.set_title('Mid-chain Entanglement vs Transverse Field')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Peak entanglement at the critical point h/J = 1 signals the quantum phase transition")

### 2b. Single-site vs Two-site DMRG

Two-site DMRG can grow bond dimensions dynamically but uses more memory.
Single-site DMRG is more memory efficient but can get stuck in local minima.

In [ ]:
# Compare single-site and two-site DMRG convergence
n_sites = 10

dmrg_2s = DMRG.ising(n_sites=n_sites, J=1.0, h=0.5, max_bond_dim=16)
e_2s, _ = dmrg_2s.run(n_sweeps=20, algorithm='2site')
hist_2s = dmrg_2s.energy_history

dmrg_1s = DMRG.ising(n_sites=n_sites, J=1.0, h=0.5, max_bond_dim=16)
e_1s, _ = dmrg_1s.run(n_sweeps=20, algorithm='1site')
hist_1s = dmrg_1s.energy_history

plt.figure(figsize=(8, 4))
plt.plot(hist_2s, 'o-', label=f'2-site (E={e_2s:.6f})', markersize=4)
plt.plot(hist_1s, 's-', label=f'1-site (E={e_1s:.6f})', markersize=4)
plt.xlabel('Sweep')
plt.ylabel('Energy')
plt.title(f'DMRG Convergence: Ising N={n_sites}, h/J=0.5')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 3. Custom Hamiltonians with MPOBuilder

For Hamiltonians beyond the built-in models, use `MPOBuilder` to construct an arbitrary sum of local operators.

In [ ]:
from qforge.mpo_builder import MPOBuilder

# Build a custom Hamiltonian: dimerized Heisenberg chain
# H = sum_i J_i (X_i X_{i+1} + Y_i Y_{i+1} + Z_i Z_{i+1})
# where J_i alternates between J1=1.0 (strong) and J2=0.5 (weak)
n_sites = 8
J1, J2 = 1.0, 0.5

builder = MPOBuilder(n_sites)
for i in range(n_sites - 1):
    J = J1 if i % 2 == 0 else J2
    builder.add_term(J, [('X', i), ('X', i + 1)])
    builder.add_term(J, [('Y', i), ('Y', i + 1)])
    builder.add_term(J, [('Z', i), ('Z', i + 1)])

# Add a staggered magnetic field
h_stag = 0.1
for i in range(n_sites):
    sign = 1.0 if i % 2 == 0 else -1.0
    builder.add_term(sign * h_stag, [('Z', i)])

# Solve with DMRG
dmrg = DMRG.custom(n_sites, builder=builder, max_bond_dim=32, backend='python')
energy, psi = dmrg.run(n_sweeps=15, verbose=False)
print(f"Dimerized Heisenberg + staggered field:")
print(f"  Ground state energy: {energy:.6f}")
print(f"  Energy per site:     {energy / n_sites:.6f}")
print(f"  Terms in Hamiltonian: {len(builder.terms)}")

---
## 4. Excited States

Find multiple eigenstates using the penalty method: after finding the ground state, penalize overlap with it to find the first excited state, and so on.

In [ ]:
# Find the first 4 eigenstates of the transverse-field Ising model
n_sites = 6  # small for exact comparison
dmrg = DMRG.ising(n_sites=n_sites, J=1.0, h=1.0, max_bond_dim=16)

results = dmrg.run_excited(n_states=4, n_sweeps=15, verbose=False, weight=20.0)

print(f"Ising chain N={n_sites}, h/J=1.0 — first 4 eigenstates:")
print(f"{'State':>6} {'Energy':>12} {'Gap':>10}")
print("-" * 32)
e0 = results[0][0]
for i, (e, psi) in enumerate(results):
    gap = e - e0
    print(f"{i:>6} {e:>12.6f} {gap:>10.6f}")

# Compare to exact diagonalization
from qforge.mpo_builder import _PAULI
I2 = np.eye(2, dtype=complex)
Xm, Zm = _PAULI['X'], _PAULI['Z']
H_exact = np.zeros((2**n_sites, 2**n_sites), dtype=complex)
for i in range(n_sites - 1):
    op = np.eye(1)
    for j in range(n_sites):
        op = np.kron(op, Zm if j in (i, i+1) else I2)
    H_exact -= op
for i in range(n_sites):
    op = np.eye(1)
    for j in range(n_sites):
        op = np.kron(op, Xm if j == i else I2)
    H_exact -= op

exact_eigs = np.sort(np.linalg.eigvalsh(H_exact))[:4]
print(f"\nExact eigenvalues: {exact_eigs}")
print(f"DMRG energies:     {[r[0] for r in results]}")

---
## 5. TEBD — Time Evolution

TEBD (Time-Evolving Block Decimation) simulates quantum dynamics by decomposing the time-evolution operator into a sequence of two-site gates via Trotter-Suzuki decomposition.

### 5a. Quench dynamics

Start from a product state and evolve under a Heisenberg Hamiltonian. Watch entanglement grow.

In [ ]:
from qforge.tebd import TEBD

# Prepare a Neel state: |0101...>
n_qubits = 20
psi = MatrixProductState(n_qubits=n_qubits, max_bond_dim=64)
for i in range(1, n_qubits, 2):
    X(psi, i)  # flip odd qubits to |1>

# Set up Heisenberg time evolution
tebd = TEBD.heisenberg(psi, J=1.0, dt=0.05, order=2)

# Evolve and track mid-chain entanglement entropy
times = []
entropies = []
bond_dims = []

print("Quench: Neel state under Heisenberg dynamics")
for step in range(60):
    t = step * 0.05
    times.append(t)
    entropies.append(psi.entanglement_entropy(n_qubits // 2 - 1))
    bond_dims.append(max(psi.bond_dimensions()))
    tebd.step(1)
    if step % 20 == 0:
        print(f"  t={t:.2f}: S_mid={entropies[-1]:.4f} bits, max chi={bond_dims[-1]}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(times, entropies, linewidth=2)
ax1.set_xlabel('Time t')
ax1.set_ylabel('Entanglement entropy (bits)')
ax1.set_title('Mid-chain Entanglement Growth After Quench')
ax1.grid(True, alpha=0.3)

ax2.plot(times, bond_dims, linewidth=2, color='darkorange')
ax2.set_xlabel('Time t')
ax2.set_ylabel('Max bond dimension')
ax2.set_title('Bond Dimension Growth')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5b. Imaginary-time TEBD for ground states

Imaginary-time evolution $e^{-\tau H}$ projects onto the ground state. This is an alternative to DMRG.

In [ ]:
# Imaginary-time TEBD to find the Ising ground state
n_qubits = 12
psi = MatrixProductState(n_qubits=n_qubits, max_bond_dim=32)

# Start from random-ish state (break symmetry)
for i in range(n_qubits):
    RY(psi, i, np.random.uniform(0, np.pi))

tebd = TEBD.ising(psi, J=1.0, h=1.0, dt=0.1, order=2, imaginary=True)

# Cool down in stages (simulated annealing of dt)
energies_imag = []
for dt, n_steps in [(0.1, 30), (0.01, 50), (0.001, 50)]:
    tebd.dt = dt
    tebd._precompute_gates(dt)
    for _ in range(n_steps):
        tebd.step(1)
        energies_imag.append(tebd.energy())

# Compare to DMRG
dmrg_ref = DMRG.ising(n_sites=n_qubits, J=1.0, h=1.0, max_bond_dim=32)
e_dmrg, _ = dmrg_ref.run(n_sweeps=15)

plt.figure(figsize=(8, 4))
plt.plot(energies_imag, linewidth=2, label='Imaginary-time TEBD')
plt.axhline(y=e_dmrg, color='red', linestyle='--', label=f'DMRG E={e_dmrg:.4f}')
plt.xlabel('Imaginary-time step')
plt.ylabel('Energy')
plt.title(f'Ground State via Imaginary TEBD (Ising N={n_qubits})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final TEBD energy: {energies_imag[-1]:.6f}")
print(f"DMRG energy:       {e_dmrg:.6f}")

---
## 6. iTEBD — Infinite Systems

iTEBD simulates **infinite** 1D systems directly using a two-site unit cell in Vidal canonical form. No finite-size effects!

### 6a. Ground state of the infinite Ising chain

In [ ]:
from qforge.itebd import iTEBD

# Ising model at the critical point h/J = 1
sim = iTEBD.ising(J=1.0, h=1.0, chi=64)

# Imaginary-time evolution: decrease dt for precision
convergence = []
for dt, n_steps in [(0.1, 50), (0.01, 100), (0.001, 100)]:
    for _ in range(n_steps):
        sim.evolve_imaginary(dt=dt, n_steps=1, order=2)
        convergence.append(sim.energy())

print(f"Infinite Ising chain at h/J = 1.0 (critical point):")
print(f"  Energy per site:      {sim.energy():.8f}")
print(f"  Exact (Pfeuty 1970):  {-4/np.pi:.8f}")
print(f"  Error:                {abs(sim.energy() - (-4/np.pi)):.2e}")
print(f"  Entanglement entropy: {sim.entanglement_entropy():.4f} bits")
print(f"  Correlation length:   {sim.correlation_length():.2f} sites")
print(f"  Bond dimension:       {sim.bond_dimension()}")

plt.figure(figsize=(8, 4))
plt.plot(convergence, linewidth=1)
plt.axhline(y=-4/np.pi, color='red', linestyle='--', label=f'Exact: {-4/np.pi:.6f}')
plt.xlabel('iTEBD step')
plt.ylabel('Energy per site')
plt.title('iTEBD Convergence: Infinite Ising Chain')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6b. Phase diagram scan

Sweep `h/J` and plot the energy, entanglement, and correlation length to map out the phase diagram.

In [ ]:
# Phase diagram via iTEBD
h_vals = np.linspace(0.1, 2.5, 20)
itebd_energies = []
itebd_entropy = []
itebd_xi = []

for h_val in h_vals:
    sim = iTEBD.ising(J=1.0, h=h_val, chi=32)
    sim.evolve_imaginary(dt=0.1, n_steps=50, order=2)
    sim.evolve_imaginary(dt=0.01, n_steps=80, order=2)

    itebd_energies.append(sim.energy())
    itebd_entropy.append(sim.entanglement_entropy())
    xi = sim.correlation_length()
    itebd_xi.append(min(xi, 100))  # cap for plotting

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(h_vals, itebd_energies, 'o-', markersize=4, color='navy')
axes[0].axvline(x=1.0, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('h / J')
axes[0].set_ylabel('Energy per site')
axes[0].set_title('Ground State Energy (iTEBD)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(h_vals, itebd_entropy, 's-', markersize=4, color='darkgreen')
axes[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('h / J')
axes[1].set_ylabel('Entanglement entropy (bits)')
axes[1].set_title('Entanglement (iTEBD)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(h_vals, itebd_xi, '^-', markersize=4, color='darkorange')
axes[2].axvline(x=1.0, color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('h / J')
axes[2].set_ylabel('Correlation length (sites)')
axes[2].set_title('Correlation Length (iTEBD)')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Infinite Ising Chain Phase Diagram', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The critical point at h/J=1 shows: diverging correlation length, peak entanglement")

---
## 7. Distributed Simulation (MPI)

For exact simulation beyond ~22 qubits, Qforge can shard the state vector across MPI processes. Each rank holds a contiguous block of amplitudes; gates on "local" qubits require no communication, while "global" qubits use pairwise MPI exchange.

### Setup

```bash
# Build with MPI support
QFORGE_MPI=1 pip install -e .

# Run a circuit across 4 processes (enables ~2 extra qubits)
mpirun -np 4 python my_circuit.py
```

### Scale reference

| Qubits | Ranks | Memory/rank | Total memory |
|--------|-------|-------------|-------------|
| 25     | 2     | 256 MB      | 512 MB      |
| 28     | 4     | 1 GB        | 4 GB        |
| 30     | 4     | 4 GB        | 16 GB       |
| 33     | 8     | 8 GB        | 64 GB       |
| 36     | 64    | 8 GB        | 512 GB      |

### Usage

```python
from qforge.distributed import DistributedQubit
from qforge.gates import H, CNOT
from qforge.measurement import measure_one

# Works exactly like Qubit() — same gate API
wf = DistributedQubit(30)
H(wf, 0)
for i in range(29):
    CNOT(wf, i, i + 1)
print(measure_one(wf, 0))  # [0.5, 0.5]
```

> **Note**: The distributed backend requires MPI and is not demonstrated interactively in this notebook. See `tests/test_distributed.py` for a full example.

---
## 8. Choosing the Right Backend

| Scenario | Backend | Qubits | Key function |
|----------|---------|--------|-------------|
| Small circuits, full access to amplitudes | `Qubit(n)` | 1-22 | State vector |
| Large low-entanglement circuits | `MatrixProductState(n, chi)` | 2-1000+ | MPS |
| Ground states of 1D Hamiltonians | `DMRG.ising(...)` | 2-1000+ | DMRG sweeps |
| Custom Hamiltonians | `DMRG.custom(builder=...)` | 2-100+ | MPO builder |
| Quantum dynamics | `TEBD(psi, bonds, dt)` | 2-100+ | Trotter evolution |
| Infinite systems (no boundary) | `iTEBD.ising(...)` | Infinite | Vidal form |
| Large exact circuits on cluster | `DistributedQubit(n)` | 25-40 | MPI sharding |

### Quick decision tree

1. **Need exact amplitudes?** Use `Qubit(n)` for n <= 22, `DistributedQubit(n)` for n <= 40
2. **Ground state of a 1D Hamiltonian?** Use `DMRG` (finite) or `iTEBD` (infinite)
3. **Time evolution?** Use `TEBD` (finite chain) or `iTEBD` (infinite chain)
4. **Low-entanglement circuit with many qubits?** Use `MatrixProductState`